# 09:45 双 ETF 分钟路径分歧的来源预检

只读封存 Qlib 1min 来源与两个 Recorder。固定15分钟、固定公式及覆盖/新颖性门；无未来收益、拟合或账户读取。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R

root=Path.cwd();card=root/"research/stage1_0945_pair_path_divergence_source_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["source"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["budget"]["new_market_GETs"]==plan["budget"]["SOXX_outcome_reads"]==plan["budget"]["model_fits"]==plan["budget"]["native_accounts"]==0
assert plan["budget"]["Qlib_feature_attempts_max"]==2
assert sha(root/src["provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
manifest=json.loads((root/src["provider"]/"qlib_store_manifest.json").read_text())
assert manifest["freq"]=="1min" and manifest["instruments"]==["SOXL","SOXX"]
assert all(x in manifest["fields"] for x in ["$open","$close","$price_valid"])
parent=root/src["parent_card"];f696=root/src["f696_card"]
assert sha(parent/"summary.json")==src["parent_summary_sha256"]
assert sha(parent/"nas_readback.json")==src["parent_nas_readback_sha256"]
assert sha(parent/"nas_binding.json")==src["parent_nas_binding_sha256"]
assert sha(f696/"summary.json")==src["f696_summary_sha256"]
assert sha(root/src["prior_closed_cross_response_card"]/"summary.json")==src["prior_closed_cross_response_summary_sha256"]
ps=json.loads((parent/"summary.json").read_text());pr=json.loads((parent/"nas_readback.json").read_text());pb=json.loads((parent/"nas_binding.json").read_text());fs=json.loads((f696/"summary.json").read_text())
assert ps["nas_evidence"]["sha256"]==src["parent_archive_sha256"] and ps["nas_evidence"]["size_bytes"]==src["parent_archive_size_bytes"]
assert pr["NAS_archive_sha256"]==pb["evidence"]["sha256"]==src["parent_archive_sha256"] and pb["verified"]
assert fs["nas_evidence"]["sha256"]==src["f696_archive_sha256"] and fs["nas_evidence"]["size_bytes"]==src["f696_archive_size_bytes"]
pa=parent/"qlib_recorder"/src["parent_experiment_id"]/src["parent_recorder_id"]/"artifacts"
fa=f696/"qlib_recorder"/src["f696_experiment_id"]/src["f696_recorder_id"]/"artifacts"
for name,digest in src["parent_artifacts"].items():assert sha(pa/name)==digest,name
assert sha(fa/"inputs/frame")==src["f696_frame_sha256"]
qlib.init(provider_uri=str((root/src["provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str(parent/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["parent_experiment_id"],recorder_id=src["parent_recorder_id"])
 features=rec.load_object("inputs/features")
 daily_frame=rec.load_object("inputs/daily_frame")
with R.uri_context(str(f696/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["f696_experiment_id"],recorder_id=src["f696_recorder_id"])
 frame=rec.load_object("inputs/frame")
assert len(features)==len(daily_frame)==3966 and len(frame)==944
original=ps["input_audit"]["original_features"]
assert len(original)==39 and list(features.columns[:39])==original
minute_calendar=pd.DatetimeIndex(D.calendar(start_time="2018-10-12 09:31:00",end_time="2026-10-09 16:00:00",freq="1min"))
assert len(minute_calendar)>100000 and minute_calendar.is_monotonic_increasing
bars=D.features(["SOXL","SOXX"],["$open","$close","$price_valid"],start_time="2018-10-12 09:31:00",end_time="2026-10-09 16:00:00",freq="1min").reset_index()
assert {"instrument","datetime","$open","$close","$price_valid"}.issubset(bars.columns)
bars["datetime"]=pd.to_datetime(bars.datetime)
clock=bars.datetime.dt.strftime("%H:%M")
opening=bars.loc[clock.between("09:31","09:45")].copy()
opening["day"]=opening.datetime.dt.normalize()
rows=[];invalid=[]
for day,g in opening.groupby("day",sort=True):
 expected=pd.date_range(day+pd.Timedelta(hours=9,minutes=31),periods=15,freq="min")
 parts={sym:g.loc[g.instrument.eq(sym)].sort_values("datetime") for sym in ("SOXL","SOXX")}
 valid=True
 for sym,x in parts.items():
  valid &= len(x)==15 and pd.DatetimeIndex(x.datetime).equals(expected)
  if len(x)==15:
   valid &= bool(np.isfinite(x[["$open","$close","$price_valid"]].to_numpy(float)).all())
   valid &= bool((x["$open"]>0).all() and (x["$close"]>0).all() and x["$price_valid"].eq(1).all())
 if not valid:
  invalid.append(str(day.date()));continue
 sx,xx=parts["SOXL"],parts["SOXX"]
 rl=np.log(sx["$close"].to_numpy(float)/sx["$open"].to_numpy(float))
 rx=np.log(xx["$close"].to_numpy(float)/xx["$open"].to_numpy(float))
 energy=float(np.square(rl-3*rx).sum()/(np.square(rl).sum()+9*np.square(rx).sum()+1e-12))
 endpoint=float(sx["$close"].iloc[-1]/sx["$open"].iloc[0]-1-3*(xx["$close"].iloc[-1]/xx["$open"].iloc[0]-1))
 rows.append({"trade_date":day,"pair_path_divergence":energy,"endpoint_cross_response":endpoint,
              "early_ret_soxl":float(sx["$close"].iloc[-1]/sx["$open"].iloc[0]-1),
              "early_ret_soxx":float(xx["$close"].iloc[-1]/xx["$open"].iloc[0]-1)})
candidate=pd.DataFrame(rows).set_index("trade_date").sort_index()
assert candidate.index.is_unique and candidate.pair_path_divergence.between(0,2.01).all()
train_map=daily_frame.loc[pd.to_datetime(daily_frame.trade_date).between(pd.Timestamp("2018-10-12"),pd.Timestamp("2022-12-30")),["trade_date"]].copy()
train_map["signal_date"]=pd.DatetimeIndex(train_map.index)
train_map["trade_date"]=pd.to_datetime(train_map.trade_date).dt.normalize()
held_map=frame.loc[:,["signal_date","trade_date"]].copy()
held_map["signal_date"]=pd.to_datetime(held_map.signal_date)
held_map["trade_date"]=pd.to_datetime(held_map.trade_date).dt.normalize()
assert train_map.trade_date.is_unique and held_map.trade_date.is_unique and len(held_map)==944
known_train=train_map.trade_date.isin(candidate.index)
known_held=held_map.trade_date.isin(candidate.index)
train=train_map.loc[known_train].copy();held=held_map.loc[known_held].copy()
train["pair_path_divergence"]=candidate.loc[train.trade_date,"pair_path_divergence"].to_numpy()
train["endpoint_cross_response"]=candidate.loc[train.trade_date,"endpoint_cross_response"].to_numpy()
train["early_ret_soxl"]=candidate.loc[train.trade_date,"early_ret_soxl"].to_numpy()
train["early_ret_soxx"]=candidate.loc[train.trade_date,"early_ret_soxx"].to_numpy()
feature_train=features.reindex(pd.DatetimeIndex(train.signal_date))
assert feature_train.index.equals(pd.DatetimeIndex(train.signal_date))
# Keep the fixed training values aligned by row rather than pandas index labels.
source_values=train.pair_path_divergence.reset_index(drop=True)
novelty={name:float(source_values.corr(feature_train[name].reset_index(drop=True),method="spearman")) for name in original}
tracking=float(source_values.corr(feature_train["minute_soxl_tracking_residual_rv5"].reset_index(drop=True),method="spearman"))
endpoint=float(source_values.corr(train.endpoint_cross_response.reset_index(drop=True),method="spearman"))
early={name:float(source_values.corr(train[name].reset_index(drop=True),method="spearman")) for name in ["early_ret_soxl","early_ret_soxx"]}
std=float(train.pair_path_divergence.std(ddof=0))
coverage={"train_expected":len(train_map),"train_known":int(known_train.sum()),"held_expected":len(held_map),"held_known":int(known_held.sum()),
          "train_unknown_trade_dates":[str(x.date()) for x in train_map.loc[~known_train,"trade_date"]],
          "held_unknown_trade_dates":[str(x.date()) for x in held_map.loc[~known_held,"trade_date"]]}
gates={"training_support":coverage["train_known"]>=1000,"held_support":coverage["held_known"]>=900,
       "variation":std>0.02,"novel_original39":max(abs(v) for v in novelty.values())<0.80,
       "novel_prior_tracking":abs(tracking)<0.80,"novel_closed_endpoint":abs(endpoint)<0.80}
result={"status":"source_and_novelty_preflight_pass" if all(gates.values()) else "source_or_novelty_preflight_failed",
        "plan_sha256":sha(plan_path),"provider_manifest_sha256":src["provider_manifest_sha256"],"parent_archive_sha256":src["parent_archive_sha256"],
        "minute_calendar_rows":len(minute_calendar),"raw_Qlib_rows":len(bars),"qualified_pair_dates":len(candidate),
        "invalid_pair_dates":invalid,"coverage":coverage,"training_std":std,
        "training_max_abs_spearman_original39":max(abs(v) for v in novelty.values()),
        "training_spearman_prior_tracking_residual_magnitude":tracking,
        "training_spearman_closed_endpoint_cross_response":endpoint,
        "training_spearman_early_returns":early,"gates":gates,
        "historical_first_public_qualified":False,"source_clock":"assumed 09:45 bar complete before 09:46 action, not historical first-receipt proof",
        "new_market_GETs":0,"SOXX_outcome_reads":0,"model_fits":0,"native_accounts":0,"new_Recorders":0,
        "factor_economic_value_tested":False,"independent_validation":False,
        "limits":["SOXL and SOXX track different benchmarks; this is cross-ETF path divergence, not true fund tracking error",
                  "Retrospective Qlib minute provider and source receipt do not certify every historical 09:46 first-public value",
                  "Coverage and low feature correlation do not prove predictive information or SOXX account value"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"status":result["status"],"coverage":coverage,"std":std,"max_original39":result["training_max_abs_spearman_original39"],"tracking":tracking,"endpoint":endpoint,"gates":gates})


{'status': 'source_and_novelty_preflight_pass', 'coverage': {'train_expected': 1062, 'train_known': 1061, 'held_expected': 944, 'held_known': 944, 'train_unknown_trade_dates': ['2020-03-16'], 'held_unknown_trade_dates': []}, 'std': 0.15677489534098663, 'max_original39': 0.3553472689169119, 'tracking': 0.04388337103307739, 'endpoint': 0.04304259842400328, 'gates': {'training_support': True, 'held_support': True, 'variation': True, 'novel_original39': True, 'novel_prior_tracking': True, 'novel_closed_endpoint': True}}


## 中文结论

用 NAS 已核验来源构建的封存 Qlib 1 分钟 SOXL/SOXX 数据，只读取每个交易日09:31–09:45的15对完整、正价、有效分钟。固定分歧强度为两只ETF每分钟对数涨跌 `r_SOXL−3r_SOXX` 的平方和，除以两边分钟波动平方和；由于两只ETF跟踪的指数不同，**不称它为SOXL真实跟踪误差**。它仅能在假定09:45末根完成后供09:46研究决策使用。

训练期2018–2022对应的1062个Stage1交易日中 **1061日**来源已知，2020-03-16保持未知；精确f696评价期 **944/944日**已知。训练标准差0.15677。固定训练样本的最大绝对Spearman：对原39项 **0.35535**，对旧的前一日全天残差波动幅度 **0.04388**，对已失败的同段合计涨跌响应 **0.04304**；来源覆盖、变化幅度和预先冻结的重复性门全部通过。这只说明该分钟路径形状不是旧合计响应的同义替换，不说明它预测收益。

决定 `source_and_novelty_pass_needs_separate_0946_local_value_information_and_first_receipt`。下一步只能另行冻结**同状态、含0bp额外滑点且仍扣Tiered费用的09:46局部资金动作价值**信息检验，并纳入原39项、账户状态与首段涨跌的匹配对照。当前Qlib历史为回溯数据，旧日09:46首次送达未认证；即使信息检验通过，也不能直接晋升策略或称独立前向日。0未来SOXX结果读取、0拟合、0账户，SOXX目标未达成。

首次单日Qlib样本从stdin运行触发macOS子进程无法重开`<stdin>`，没有返回因子值；保留初始计划与`transport_repair.json`。正式notebook将Qlib设为单进程，只执行一次完整来源预检，未改公式、窗口或门槛。
